# **Fine Tunning**  👨🏻‍💻

In [1]:
#Conection with Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
dataset_amazon_titles = "/content/drive/MyDrive/DatasetAmazonTitles/trn.json"
amazon_titles_alpaca_format_path = "/content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_format_v3.jsonl"

In [3]:
import json
import random

instructions = [
    "Describe a product by title",
    "Describe this product",
    "Give a description of the product type",
    "Explain what this product is",
    "Provide details about this product",
    "Sumarize the type of this product"
    ]

with open(dataset_amazon_titles, "r", encoding="utf-8") as f, open(amazon_titles_alpaca_format_path, "w", encoding="utf-8") as out:
  for line in f:
    data = json.loads(line)
    title = data.get("title")
    content = data.get("content")

    if not title or not content:
      continue

    dict_alpaca_style = {
        "Instruction": random.choice(instructions),
        "Input": title.strip(),
        "Output": content.strip()
    }

    out.write(json.dumps(dict_alpaca_style, ensure_ascii=False) + "\n")


In [4]:
# check the new file

!head -n 10 /content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_format_v3.jsonl

{"Instruction": "Provide details about this product", "Input": "Girls Ballet Tutu Neon Pink", "Output": "High quality 3 layer ballet tutu. 12 inches in length"}
{"Instruction": "Describe a product by title", "Input": "Mog's Kittens", "Output": "Judith Kerr&#8217;s best&#8211;selling adventures of that endearing (and exasperating) cat Mog have entertained children for more than 30 years. Now, even infants and toddlers can enjoy meeting this loveable feline. These sturdy little board books&#8212;with their bright, simple pictures, easy text, and hand&#8211;friendly formats&#8212;are just the thing to delight the very young. Ages 6 months&#8211;2 years."}
{"Instruction": "Describe a product by title", "Input": "Girls Ballet Tutu Neon Blue", "Output": "Dance tutu for girls ages 2-8 years. Perfect for dance practice, recitals and performances, costumes or just for fun!"}
{"Instruction": "Provide details about this product", "Input": "The Prophet", "Output": "In a distant, timeless place, a 

In [6]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformer<0.9.0" peft accelerate bitsandbytes

In [7]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.4: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [17]:
from unsloth import to_sharegpt
from datasets import load_dataset

dataset_path = "/content/drive/MyDrive/DatasetAmazonTitles/amazon_titles_alpaca_format_v3.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(dataset.column_names)
print(dataset[0])

['Instruction', 'Input', 'Output']
{'Instruction': 'Provide details about this product', 'Input': 'Girls Ballet Tutu Neon Pink', 'Output': 'High quality 3 layer ballet tutu. 12 inches in length'}


In [23]:
from unsloth import to_sharegpt

dataset = to_sharegpt(
    dataset,
    merged_prompt="{Instruction}[[\nYour input is:\n{Input}]]",
    output_column_name="Output",
    conversation_extension=3,  # Select more to handle longer conversations
)

Merging columns:   0%|          | 0/1390403 [00:00<?, ? examples/s]

Converting to ShareGPT:   0%|          | 0/1390403 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/1390403 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/1390403 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/1390403 [00:00<?, ? examples/s]

Extending conversations:   0%|          | 0/1390403 [00:00<?, ? examples/s]

In [26]:
from unsloth import standardize_sharegpt

dataset = standardize_sharegpt(dataset)

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/1390403 [00:00<?, ? examples/s]

In [28]:
from unsloth import apply_chat_template

dataset = apply_chat_template(
    dataset,
    tokenizer=tokenizer,
    # default_system_message = "You are a helpful assistant", << [OPTIONAL]
)

Unsloth: Base llama-3 models did not train <|eot_id|>.
Please use the instruct version or use <|end_of_text|>
Unsloth: We automatically added an EOS token to stop endless generations.


Map:   0%|          | 0/1390403 [00:00<?, ? examples/s]